# Becke Partition 一阶梯度简单理解

In [4]:
from pyscf import gto, dft, lib, grad, hessian, data
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [5]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [6]:
def get_grids(xyz):
    mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()
    grids = dft.grid.Grids(mol).build()
    return mol, grids

## PySCF 解析与数值导数验证

需要留意，PySCF 先前使用的 `grids_response_cc` 似乎是由于后来引入 padding 的问题，其结果我不太确定是否正确。目前使用的是 PySCF 中用于计算 VV10 导数所用到的函数。

In [7]:
xyz_0 = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_p = """
N  0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""
xyz_m = """
N -0.0001   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

In [12]:
mol, grids = get_grids(xyz_0)

一阶解析格点导数计算如下：

In [13]:
dw = hessian.rks.get_dweight_dA(mol, grids)
dw.shape

(4, 3, 43328)

作为例子，一阶数值导数的第一个分量计算如下：

In [16]:
tmp_num_d = (get_grids(xyz_p)[1].weights - get_grids(xyz_m)[1].weights) / (2 * 0.0001 / data.nist.BOHR)

解析与数值导数有强一致。

In [17]:
np.allclose(dw[0, 0], tmp_num_d)

True

## Becke Partition 一阶梯度实现与公式对应